# Basic Data Exploration and Cleaning using Pandas

Dataset: Myntra product records (`Combined_dataset.csv`).  
We use `initial_price` as **price** and `ratings_count` as **quantity** to compute `total_amount = price * quantity`.

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")
RAW_CSV = DATA_DIR / "Combined_dataset.csv"
CLEAN_CSV = DATA_DIR / "cleaned_dataset.csv"

## 1. Load Data

In [ ]:
df = pd.read_csv(RAW_CSV)
print(f"Loaded {len(df)} rows and {len(df.columns)} columns from {RAW_CSV.name}")

## 2. Explore Data

In [ ]:
print("First 5 rows:")
df.head()

In [ ]:
print("Last 5 rows:")
df.tail()

In [ ]:
print(f"Shape (rows, columns): {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
print(list(df.columns))

In [ ]:
print("Data types:")
df.dtypes

In [ ]:
df.describe(include="all").T.head(12)

## 3. Handle Missing Values

In [ ]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0])
print(f"\nTotal missing cells: {df.isnull().sum().sum()}")

In [ ]:
df_clean = df.copy()

# Fill numeric missing values
df_clean["discount"] = df_clean["discount"].fillna(0)
df_clean["rating"] = df_clean["rating"].fillna(df_clean["rating"].median())

# Fill text missing values
df_clean["seller_name"] = df_clean["seller_name"].fillna("Unknown")
df_clean["what_customers_said"] = df_clean["what_customers_said"].fillna("No reviews")

print("Missing values after cleaning (columns that had gaps):")
cols_with_gaps = ["discount", "what_customers_said", "seller_name", "videos", "seller_information", "variations"]
print(df_clean[cols_with_gaps].isnull().sum())

## 4. Basic Operations (Filter & Select Columns)

In [ ]:
selected = df_clean[["product_id", "title", "initial_price", "ratings_count", "rating", "category"]]
print("Selected columns (sample):")
selected.head()

In [ ]:
# Filter: backpacks with rating >= 4.5 and price (initial_price) > 2000
filtered = df_clean[
    (df_clean["category"] == "backpacks")
    & (df_clean["rating"] >= 4.5)
    & (df_clean["initial_price"] > 2000)
]
print(f"Filtered rows: {len(filtered)}")
filtered[["product_id", "title", "initial_price", "rating", "category"]].head(10)

## 5. Remove Duplicates

In [ ]:
print(f"Rows before drop_duplicates: {len(df_clean)}")
print(f"Duplicate rows: {df_clean.duplicated().sum()}")
print(f"Duplicate product_id: {df_clean.duplicated(subset=['product_id']).sum()}")

df_clean = df_clean.drop_duplicates(subset=["product_id"], keep="first")
print(f"Rows after removing duplicate product_id: {len(df_clean)}")

## 6. Create Derived Column: total_amount

In [ ]:
df_clean["price"] = df_clean["initial_price"]
df_clean["quantity"] = df_clean["ratings_count"]
df_clean["total_amount"] = df_clean["price"] * df_clean["quantity"]

df_clean[["title", "price", "quantity", "total_amount"]].head(10)

## 7. Save Cleaned Data

In [ ]:
df_clean.to_csv(CLEAN_CSV, index=False)
print(f"Saved cleaned data to: {CLEAN_CSV.resolve()}")
print(f"Final shape: {df_clean.shape}")